# Глава 12: Обучение вашей модели

> «Я не терпел неудачи. Я просто нашёл 10 000 способов, которые не работают.»
> — **Томас Эдисон**, изобретатель

---

## Что вы узнаете

- Как предсказание следующего токена (next-token prediction) учит модели писать
- 5-шаговый рецепт обучения, который используется во всех нейронных сетях
- Как кривые потерь показывают, обучается ли ваша модель
- Как предотвратить переобучение (overfitting) и понять, когда остановиться
- Сохранение контрольных точек (checkpoints), чтобы никогда не потерять прогресс
- Волнение от наблюдения за тем, как ваша модель превращается из бессмыслицы в связный текст

---

## Установка

Сначала установим необходимые пакеты:

In [ ]:
# Установка проверенных версий
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== ИМПОРТ =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Установка устройства
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Используемое устройство: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== ВОСПРОИЗВОДИМОСТЬ =====
import random
import numpy as np

def set_seed(seed=42):
    """Установка семян для всех используемых источников случайности."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

## 1. Компоненты модели из глав 10-11

Сначала подключим модель MiniGPT, которую мы построили в предыдущих главах.

In [ ]:
# ===== МНОГОГОЛОВОЕ ВНИМАНИЕ (из главы 10) =====
class MultiHeadAttention(nn.Module):
    """
    Эффективное многоголовое внимание (все головы обрабатываются вместе).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model должен делиться на num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Объединённая QKV проекция (в 3 раза эффективнее отдельных!)
        # Почему 3 * d_model? Потому что мы проецируем в Q, K, V одновременно
        # bias=True соответствует GPT-2, поэтому позже мы сможем загрузить его предобученные смещения
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Выходная проекция
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout на весах внимания
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Шаг 1: Проекция в Q, K, V (всё сразу!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Шаг 2: Разделение на Q, K, V и изменение формы для многоголового =====
        # Изменяем форму на (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Переставляем в (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Разделяем на Q, K, V: каждый имеет форму (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Шаг 3: Масштабированное скалярное произведение внимания (батчи по головам) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Применяем причинную маску, если она предоставлена
        if mask is not None:
            # Расширяем маску для голов: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Взвешенная сумма значений
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Шаг 4: Конкатенация голов =====
        # Транспонируем в (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Изменяем форму на (batch, seq, d_model), это конкатенирует головы
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Шаг 5: Финальная проекция =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Тестируем

print("MultiHeadAttention определён!")

In [ ]:
# ===== ПОЛНОСВЯЗНАЯ СЕТЬ (из главы 10) =====
class FeedForward(nn.Module):
    """
    Позиционная полносвязная сеть.
    Применяется к каждой позиции независимо (одинаковые веса для всех позиций).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768 для GPT-2 small)
            d_ff: Размерность полносвязной сети (обычно 4 * d_model = 3072)
            dropout: Вероятность dropout
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), расширение
        x = F.gelu(x, approximate="tanh")  # Нелинейность (вариант GELU из GPT-2)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), проекция обратно
        return x


# Тестируем

print("FeedForward определён!")

In [ ]:
# ===== БЛОК ТРАНСФОРМЕРА (из главы 10) =====
class TransformerBlock(nn.Module):
    """
    Полный блок трансформера с многоголовым вниманием, полносвязной сетью,
    остаточными соединениями и нормализацией слоя (стиль пре-нормализации, как в GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Размерность модели (768)
            num_heads: Количество голов внимания (12)
            d_ff: Размерность полносвязной сети (3072 = 4 * d_model)
            dropout: Вероятность dropout
        """
        super().__init__()

        # Нормализация слоя (перед каждым подслоем)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Многоголовое внимание
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Полносвязная сеть
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (применяется после каждого подслоя)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: Входные эмбеддинги (batch, seq, d_model)
            mask: Причинная маска (seq, seq) или (1, 1, seq, seq)

        Returns:
            x: Выходные эмбеддинги (batch, seq, d_model)
            attn_weights: Веса внимания (batch, heads, seq, seq)
        """
        # ===== Многоголовое внимание с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД вниманием
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Остаточное соединение

        # ===== Полносвязная сеть с остаточным соединением =====
        # Пре-нормализация: Нормализуем ПЕРЕД полносвязной сетью
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Остаточное соединение

        return x, attn_weights


# Тестируем полный блок трансформера

print("TransformerBlock определён!")

In [ ]:
# ===== КОНФИГУРАЦИЯ GPT (из главы 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Конфигурация для модели MiniGPT."""
    vocab_size: int = 50257      # Размер словаря GPT-2
    max_seq_len: int = 1024      # Максимальная длина контекста
    embed_dim: int = 768         # Размерность эмбеддинга (d_model)
    num_heads: int = 12          # Количество голов внимания
    num_layers: int = 12         # Количество блоков трансформера
    d_ff: int = 3072             # Размерность скрытого слоя полносвязной сети (4 * embed_dim)
    dropout: float = 0.1         # Вероятность дропаута

    def __post_init__(self):
        """Проверка конфигурации после инициализации."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) должен делиться на " \
            f"num_heads ({self.num_heads})"

In [ ]:
# ===== МОДЕЛЬ MINIGPT (из главы 11) =====
class MiniGPT(nn.Module):
    """
    Минимальная языковая модель в стиле GPT.
    Объединяет эмбеддинги, блоки трансформера и голову языкового моделирования.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Эмбеддинги токенов и позиций (глава 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Блоки трансформера (глава 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Финальная нормализация слоя =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Голова языкового моделирования =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Связывание весов =====
        # Общие веса между эмбеддингом и выходной проекцией
        self.lm_head.weight = self.token_embed.weight

        # Инициализация весов
        self._init_weights()

    def _init_weights(self):
        """Инициализация весов небольшими случайными значениями."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Прямой проход через модель.

        Args:
            token_ids: ID входных токенов (batch, seq)
            return_attention: Возвращать ли веса внимания

        Returns:
            logits: Оценки для словаря (batch, seq, vocab_size)
            attention_weights: Необязательный список весов внимания по блокам
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Эмбеддинги =====
        # Эмбеддинги токенов
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Позиционные эмбеддинги
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Объединяем и применяем dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Создание причинной маски =====
        # Нижнетреугольная: каждая позиция может смотреть только на предыдущие позиции
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Проход через блоки трансформера =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Финальная нормализация слоя =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Голова языкового моделирования =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Генерация токенов, продолжающих input_ids (промпт включён в вывод).

        Args:
            input_ids: ID токенов промпта (batch, seq)
            max_new_tokens: Сколько новых токенов сгенерировать
            temperature: Температура сэмплирования (используется только при do_sample=True)
            do_sample: Сэмплировать из распределения; False = жадный argmax
            eos_token_id: Остановиться раньше, если каждая последовательность сгенерировала этот токен

        Returns:
            ID токенов (batch, seq + новые токены)
        """
        self.eval()
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids

## 2. Загрузка набора данных

Мы будем использовать TinyShakespeare — достаточно маленький набор данных, чтобы обучить модель за несколько минут.

In [ ]:
# Загрузка TinyShakespeare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Размер набора данных: {len(text):,} символов")
print(f"\nОбразец:\n{text[:500]}")

## 3. Создание набора данных и DataLoader

Обратите внимание, чего `create_lm_labels` НЕ делает: ничего не маскирует. Дополнение мы маскируем в циклах обучения и оценки через маску внимания (`targets[attention_mask[:, 1:] == 0] = -100`), поэтому настоящий EOS остаётся под контролем потерь. GPT-2 дополняет своим EOS-токеном (ID 50256): маска по значению не отличит дополнение от настоящего конца текста и молча удалила бы каждое предсказание EOS, научив модель никогда не останавливаться.


In [ ]:
def create_lm_labels(token_ids):
    """
    Создание сдвинутых входов и целей для языкового моделирования.

    Args:
        token_ids: ID токенов (batch, seq)

    Returns:
        inputs: (batch, seq-1), все токены кроме последнего
        targets: (batch, seq-1), все токены кроме первого
    """
    inputs = token_ids[:, :-1]    # Всё кроме последнего токена
    targets = token_ids[:, 1:].clone()  # .clone(): циклы помечают дополнение -100 на месте

    # Здесь нет маскирования: дополнение маскируется в циклах через маску
    # внимания, поэтому настоящий EOS (GPT-2 дополняет eos, ID 50256)
    # остаётся под контролем потерь.
    return inputs, targets


from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """Простой набор данных, возвращающий фрагменты текста."""

    def __init__(self, text, chunk_size=512):
        """
        Args:
            text: Полный текст в виде строки
            chunk_size: Сколько символов на фрагмент
        """
        self.chunks = []

        # Разбиение на фрагменты
        for i in range(0, len(text) - chunk_size, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


# Создание набора данных


def collate_fn(batch, tokenizer, max_length=128):
    """
    Токенизация и дополнение батча текстовых строк.

    Args:
        batch: Список текстовых строк
        tokenizer: Используемый токенизатор
        max_length: Максимальная длина последовательности

    Returns:
        input_ids: Дополненные ID токенов (batch, max_length)
        attention_mask: 1 для настоящих токенов, 0 для дополнения
    """
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Загрузка токенизатора (у GPT-2 по умолчанию нет pad-токена; переиспользуем end-of-text)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Создание набора данных: на TinyShakespeare фрагменты по 512 символов дают
# в среднем ~156 токенов GPT-2, поэтому обрезка max_length=128 отбрасывает
# хвост большинства фрагментов. Для предобучения это нормально: каждый
# фрагмент остаётся плотным обучающим сигналом
dataset = TextDataset(text, chunk_size=512)
print(f"Количество фрагментов: {len(dataset)}")

In [ ]:
# Создание разбиения на обучающую и валидационную выборки
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Обучающая выборка: {len(train_dataset)}, Валидационная выборка: {len(val_dataset)}")

# Создание DataLoader'ов
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Проверка одного батча
input_ids, attention_mask = next(iter(train_loader))
print(f"Форма input_ids в батче: {input_ids.shape}")

## 4. Создание модели

In [ ]:
# Маленькая конфигурация для быстрого обучения
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Количество параметров: {sum(p.numel() for p in model.parameters()):,}")

## 5. Состояние «До»: Необученная модель

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    """
    Генерация текста с контролем температуры.

    Температура:
    - < 1.0: Более сфокусированно/детерминированно
    - = 1.0: Исходное распределение
    - > 1.0: Более случайно/креативно

    Args:
        model: Обученная модель
        tokenizer: Токенизатор
        prompt: Начальный текст
        max_new_tokens: Сколько токенов сгенерировать
        temperature: Температура сэмплирования

    Returns:
        Строка сгенерированного текста
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=True: делим логиты на температуру, затем сэмплируем
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(output_ids[0])


# Генерация из необученной модели
print("ДО ОБУЧЕНИЯ (случайные веса):")
print("="*50)
prompt = "The king"
print(f"Промпт: '{prompt}'")
print(f"Результат: {generate(model, tokenizer, prompt)}")
print("\n(Случайная бессмыслица, модель ещё ничему не научилась!)")

## 6. Настройка обучения

In [ ]:
# Гиперпараметры обучения
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Всего шагов обучения: {total_steps}")

# Оптимизатор и планировщик
# Помощник групп параметров из главы 11
def get_parameter_groups(model, weight_decay=0.1):
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Помощник групп параметров из главы 11: матрицы получают затухание весов,
# смещения и векторы LayerNorm — нет
optimizer = AdamW(get_parameter_groups(model, weight_decay=0.01),
                  lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Функции обучения и оценки

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """
    Обучение в течение одной эпохи.

    Returns:
        Средние потери за эпоху
    """
    model.train()  # Включаем dropout
    total_loss = 0

    progress = tqdm(dataloader, desc="Обучение")
    for input_ids, attention_mask in progress:
        # Перенос на устройство
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Создаём входы и цели (сдвинутые на один), дополнение помечено -100
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # игнорируем дополнение в потерях

        # ===== 5-ШАГОВЫЙ РЕЦЕПТ =====

        # Шаг 1: Обнуление градиентов
        optimizer.zero_grad(set_to_none=True)

        # Шаг 2: Прямой проход
        logits = model(inputs)

        # Шаг 3: Вычисление потерь
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape обрабатывает несмежные срезы
            ignore_index=-100  # Пропускаем позиции дополнения
        )

        # Шаг 4: Обратный проход
        loss.backward()

        # Важно: обрезаем ПОСЛЕ backward(), но ДО step()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        # Шаг 5: Обновление весов
        optimizer.step()
        scheduler.step()

        # ===== КОНЕЦ 5-ШАГОВОГО РЕЦЕПТА =====

        total_loss += loss.item()

        # Обновление индикатора прогресса
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """
    Оценка модели на наборе данных.

    Returns:
        avg_loss: Средние потери кросс-энтропии
        perplexity: Экспонента потерь (интерпретируемая метрика)
    """
    model.eval()  # Отключаем dropout
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # игнорируем дополнение в потерях

        logits = model(inputs)

        # Вычисление потерь
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape обрабатывает несмежные срезы
            ignore_index=-100,
            reduction='sum'  # Сумма вместо среднего
        )

        # Считаем только токены без дополнения, чтобы перплексия была честной
        total_loss += loss.item()
        total_tokens += (targets != -100).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


def should_stop_early(val_losses, patience=3):
    """
    Проверка, стоит ли остановить обучение.

    Args:
        val_losses: Список потерь на валидации по эпохам
        patience: Сколько растущих эпох допустимо

    Returns:
        True, если нужно остановиться
    """
    if len(val_losses) < patience + 1:
        return False

    # Проверяем, растут ли последние 'patience' значений потерь
    recent = val_losses[-(patience + 1):]
    for i in range(1, len(recent)):
        if recent[i] <= recent[i-1]:
            return False

    return True

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """
    Сохранение контрольной точки обучения, включая состояния RNG для точного продолжения.

    Args:
        model: Модель
        optimizer: Оптимизатор (имеет состояние моментума)
        scheduler: Планировщик скорости обучения
        epoch: Номер текущей эпохи
        train_loss: Потери на обучении за эту эпоху
        val_loss: Потери на валидации за эту эпоху
        path: Куда сохранить
    """
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        # Состояния RNG: dropout и перемешивание должны продолжить ту же последовательность
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
    }, path)
    print(f"Контрольная точка сохранена: {path}")


def load_checkpoint(model, optimizer, scheduler, path, device):
    """
    Загрузка контрольной точки обучения и восстановление состояний RNG.

    Returns:
        epoch: Эпоха, с которой продолжать
        train_loss: Последние потери на обучении
        val_loss: Последние потери на валидации
    """
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Состояния RNG должны быть CPU-тензорами: map_location=device может их перенести
    random.setstate(checkpoint['rng_python'])
    np.random.set_state(checkpoint['rng_numpy'])
    torch.set_rng_state(checkpoint['rng_torch'].cpu())
    if checkpoint['rng_cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['rng_cuda']])
    if checkpoint['rng_mps'] is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(checkpoint['rng_mps'].cpu())

    print(f"Контрольная точка загружена с эпохи {checkpoint['epoch']}")

    return checkpoint['epoch'], checkpoint['train_loss'], checkpoint['val_loss']

Один компромисс: `weights_only=True` отказывается выполнять произвольный pickle-код при загрузке — правильная привычка для чекпоинтов из недоверенных источников. Но состояние RNG NumPy содержит скаляры NumPy, которые безопасный загрузчик отклоняет, поэтому эта версия с точным продолжением использует `weights_only=False`. Загружайте так только чекпоинты, созданные вами (или явно доверенные); для всего скачанного предпочитайте `weights_only=True` и примите, что состояния RNG не восстановятся. Ещё два условия для побитово точного продолжения: пересоздайте планировщик с тем же горизонтом общего числа шагов, что и в исходном запуске, и потребляйте батчи в том же порядке.

## 8. Цикл обучения

In [ ]:
# Обучение!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Эпоха {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Обучение
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Оценка
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nПотери на обучении: {train_loss:.4f}")
    print(f"Потери на валидации:   {val_loss:.4f}")
    print(f"Перплексия: {perplexity:.1f}")

    # Сохранение лучшей контрольной точки
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nОбучение завершено!")
print(f"Лучшие потери на валидации: {best_val_loss:.4f}")

## 9. Построение кривых потерь

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Потери на обучении')
plt.plot(epochs, val_losses, 'r-o', label='Потери на валидации')
plt.xlabel('Эпоха')
plt.ylabel('Потери')
plt.title('Потери на обучении и валидации')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. Награда: Генерация текста!

In [ ]:
# Загрузка лучшей модели (наша собственная контрольная точка, weights_only=False
# здесь безопасен; состояния RNG внутри требуют полного загрузчика, см. раздел о чекпоинтах)
checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Контрольная точка загружена с эпохи {checkpoint['epoch'] + 1}")
print(f"Потери на валидации: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("ПОСЛЕ ОБУЧЕНИЯ (на текстах Шекспира):")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nПромпт: '{prompt}'")
    print(f"Результат: {output}")
    print("-"*40)

## 11. Сравнение «До» и «После»

In [ ]:
# Создание новой необученной модели для сравнения
set_seed(123)  # Другое семя для других случайных весов
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"Промпт: '{prompt}'")
print("="*60)
print("\nДО (необученная):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nПОСЛЕ (обученная на Шекспире):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nТа же архитектура. Тот же код. Обучение меняет всё!")

## 12. Сравнение температур

In [ ]:
prompt = "The noble lord"

print(f"Промпт: '{prompt}'\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Температура {temp}:")
    print(f"  {output}")
    print()

## Резюме

**Что мы построили:**

1. **Сдвиг меток** для предсказания следующего токена
2. **DataLoader'ы**, которые эффективно группируют и токенизируют текст
3. **5-шаговый рецепт обучения**: zero_grad → forward → loss → backward → step
4. **Оценку** с потерями на валидации и перплексией
5. **Контрольные точки** для сохранения и восстановления обучения
6. **Генерацию текста** с контролем температуры

**Ключевые концепции:**

- Обучение — это цикл управления с обратной связью: измерение ошибки, корректировка весов, повторение
- Перекрёстная энтропия сильнее штрафует уверенные неправильные предсказания, чем неуверенные
- Переобучение = запоминание обучающих данных (потери на обучении ↓, потери на валидации ↑)
- Перплексия измеряет «насколько модель удивлена», чем ниже, тем лучше
- Температура контролирует разнообразие генерации

**Дальше:** Глава 13 научит вас дообучать эту модель для конкретных задач!

## Упражнения

### Упражнение 1: Эксперимент со скоростью обучения

Попробуйте разные скорости обучения и сравните результаты.

In [ ]:
# ВАШ КОД ЗДЕСЬ
# Обучите с lr=1e-5, lr=3e-4, lr=1e-2
# Сравните кривые потерь
# Какая скорость обучения работает лучше всего?

### Упражнение 2: Больше эпох

In [ ]:
# ВАШ КОД ЗДЕСЬ
# Обучите в течение 5-10 эпох вместо 3
# Продолжает ли модель улучшаться?
# Видите ли вы признаки переобучения?

### Упражнение 3: Сравнение размеров модели

In [ ]:
# ВАШ КОД ЗДЕСЬ
# Создайте модель меньшего размера (2 слоя, 128 размерность)
# Создайте модель большего размера (6 слоёв, 384 размерность)
# Сравните скорость обучения и финальную перплексию